# Punishment Study 2.1.2 — NLP Pipeline
## Notebook 2: Zero-Shot Classification

**Author:** David G. Kamper  
**Project:** *Is Criminal Punishment Prosocial?*  
**Pipeline stage:** 2 of 4

---

### What this notebook does

Replaces the v2 pipeline's BART-large-MNLI classifier with two stronger models, run independently for cross-validation:

1. **DeBERTa-v3-large-zeroshot-v2.0** (`MoritzLaurer/deberta-v3-large-zeroshot-v2.0`) — open-source NLI-based zero-shot classifier, currently top of the HuggingFace zero-shot leaderboard. Free, reproducible, runs on T4.
2. **Claude Haiku 4.5** (`claude-haiku-4-5-20251001`) — LLM-based classifier accessed via the Anthropic API. Uses structured JSON output for both forced-choice and multi-label scoring.

Each classifier produces:
- **Multi-label probabilities** (independent 0–1 score per category)
- **Forced-choice top label** (single best-fitting category)
- **Prosocial/dark aggregates** (mean over the 6 prosocial vs 2 dark categories)

### A1 Benchmark validation

Both classifiers are run on the **40-sentence synthetic benchmark** (5 hand-crafted unambiguous examples × 8 categories) used to validate BART. The headline comparison for the manuscript is:

| Classifier | Top-1 accuracy | Reported in |
|---|---|---|
| BART-large-MNLI (v2 baseline) | 77.5% | (historical, for comparison) |
| DeBERTa-v3-large-zeroshot-v2.0 | TBD | this notebook |
| Claude Haiku 4.5 | TBD | this notebook |

### Pipeline context

| Notebook | Status | Output |
|---|---|---|
| 01 (preprocessing + dictionaries) | ✓ done | `01_basic_features.csv` |
| **02 (this — classification)** | running | `02_classification_features.csv` |
| 03 (embeddings + similarity) | next | `03_embedding_features.csv` |
| 04 (topics + convergence + facade + export) | last | `punishment_212_nlp_features.csv` |

### Setup

**Runtime → Change runtime type → T4 GPU** (DeBERTa benefits substantially from GPU; Claude calls are CPU/network-bound).

**API key:** This notebook reads the Anthropic API key from Colab Secrets (`Punishment_ClaudeAPI`). Add it via the key icon in the left sidebar before running.

### Runtime estimate

- DeBERTa on 496 responses (multi-label + forced-choice): ~8–12 min on T4
- Claude Haiku on 496 responses (parallelized, both modes): ~3–5 min, API cost ≈ $0.50–$0.80
- Benchmark validation on both: ~1 min


---

## Section 0: Setup


In [1]:
# === INSTALL PACKAGES ===
!pip install -q transformers==4.45.0 sentencepiece protobuf
!pip install -q anthropic
print('✓ Packages installed')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 34.6 MB/s eta 0:00:00
✓ Packages installed


In [2]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import json
import re
import time
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import pipeline
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

import anthropic
from google.colab import files, userdata

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# GPU check
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — DeBERTa will be slow')

print('✓ Imports complete')


✓ GPU available: Tesla T4
✓ Imports complete


In [3]:
# === API KEY FROM COLAB SECRETS ===
try:
    Punishment_ClaudeAPI = userdata.get('Punishment_ClaudeAPI')
    if not Punishment_ClaudeAPI:
        raise ValueError('Empty key')
    print(f'✓ Anthropic API key loaded (ends in ...{Punishment_ClaudeAPI[-4:]})')
except Exception as e:
    print(f'⚠ Could not load Punishment_ClaudeAPI from Colab secrets: {e}')
    print('  Add it via the key icon in the left sidebar.')
    raise

claude_client = anthropic.Anthropic(api_key=Punishment_ClaudeAPI)
CLAUDE_MODEL = 'claude-haiku-4-5-20251001'
print(f'✓ Claude client ready (model = {CLAUDE_MODEL})')


✓ Anthropic API key loaded (ends in ...sAAA)
✓ Claude client ready (model = claude-haiku-4-5-20251001)


---

## Section 1: Load `01_basic_features.csv` from Notebook 1


In [4]:
# === LOAD INPUT ===
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f'✓ Loaded: {filename}')
print(f'  Rows: {len(df)}, Columns: {len(df.columns)}')

# Sanity: text_combined column should be present
assert 'text_combined' in df.columns, 'Run Notebook 1 first to produce text_combined'
print(f'  Text-combined non-null: {df["text_combined"].notna().sum()}')


Saving 01_basic_features.csv to 01_basic_features.csv
✓ Loaded: 01_basic_features.csv
  Rows: 496, Columns: 202
  Text-combined non-null: 496


---

## Section 2: Categories, prosocial/dark mapping, and benchmark sentences

**Eight candidate labels** (same as v2 pipeline, kept for direct comparability with the BART results in the prior submission draft):

| Label | Group |
|---|---|
| deterrence and prevention | prosocial |
| public safety and protection | prosocial |
| rehabilitation and reform | prosocial |
| proportional justice | prosocial |
| societal condemnation | prosocial |
| victim closure | prosocial |
| punishment and suffering | dark |
| revenge and payback | dark |

Note: "victim closure" is grouped as prosocial here because it focuses on victim/family benefit. Sensitivity analyses elsewhere test whether moving it to dark or removing it changes results.

**A1 benchmark:** 5 hand-crafted unambiguous sentences per category × 8 = 40 sentences. Identical to v2 so DeBERTa and Claude scores are directly comparable to the BART 77.5% baseline.


In [5]:
# === LABELS AND GROUPING ===
LABELS = [
    'deterrence and prevention',
    'public safety and protection',
    'rehabilitation and reform',
    'proportional justice',
    'societal condemnation',
    'victim closure',
    'punishment and suffering',
    'revenge and payback',
]

PROSOCIAL_LABELS = [
    'deterrence and prevention',
    'public safety and protection',
    'rehabilitation and reform',
    'proportional justice',
    'societal condemnation',
    'victim closure',
]
DARK_LABELS = [
    'punishment and suffering',
    'revenge and payback',
]

def col(label, prefix):
    """Standardize column naming: prefix + label with spaces→underscores."""
    return f'{prefix}_' + label.replace(' ', '_')

print(f'✓ {len(LABELS)} labels defined ({len(PROSOCIAL_LABELS)} prosocial, {len(DARK_LABELS)} dark)')


✓ 8 labels defined (6 prosocial, 2 dark)


In [6]:
# === A1 SYNTHETIC BENCHMARK SENTENCES ===
BENCHMARK_SENTENCES = {
    'deterrence and prevention': [
        'This sentence will make others think twice before committing crimes.',
        'The punishment should send a clear warning to would-be offenders.',
        'By punishing this crime severely, we prevent future crimes from happening.',
        'Others who might consider committing a crime will be scared straight.',
        'A strong punishment deters criminal behavior in the community.',
    ],
    'public safety and protection': [
        'We need to keep this dangerous person away from the public.',
        'Society is safer when violent offenders are behind bars.',
        'This sentence protects innocent people from a threat.',
        'The community needs to be shielded from criminals like this.',
        'Keeping offenders locked up prevents them from hurting anyone else.',
    ],
    'rehabilitation and reform': [
        'Prison should give this person a chance to turn their life around.',
        'With proper therapy and education, this offender can become a better person.',
        'Everyone deserves a second chance to reform and reenter society.',
        'The goal of prison should be helping offenders become productive citizens.',
        'I hope this person receives treatment and learns from their mistakes.',
    ],
    'proportional justice': [
        'The punishment should fit the severity of the crime committed.',
        'This is a fair and proportionate sentence for what was done.',
        'Justice demands that the consequence match the offense.',
        'This sentence appropriately reflects how serious the crime was.',
        'The offender deserves a punishment that is neither too harsh nor too lenient.',
    ],
    'societal condemnation': [
        'Society needs to show that this behavior is completely unacceptable.',
        'This sentence sends a moral message about our values.',
        'We as a society condemn this kind of violence.',
        'The punishment expresses our collective disapproval of murder.',
        'This conviction upholds the moral standards of our community.',
    ],
    'victim closure': [
        "The victim's family deserves to see justice served.",
        'This sentence provides some peace to the loved ones affected.',
        'The family needs closure after what happened to them.',
        'Seeing the offender punished can help the victims begin to heal.',
        'This is for the family who lost someone they loved.',
    ],
    'punishment and suffering': [
        'I want him to suffer every day in prison.',
        'Prison should be as painful and miserable as possible for him.',
        'Let him rot and experience agony behind bars.',
        'He deserves to feel torment for the rest of his life.',
        'Prison should be a living hell for someone who did this.',
    ],
    'revenge and payback': [
        'He should get a taste of his own medicine.',
        'An eye for an eye. He needs to pay for what he did.',
        'This is payback for the suffering he caused.',
        'I want him to feel the same pain he inflicted on others.',
        'He deserves vengeance for the harm he brought upon his victim.',
    ],
}

n_bench = sum(len(v) for v in BENCHMARK_SENTENCES.values())
print(f'✓ Benchmark loaded: {n_bench} sentences across {len(BENCHMARK_SENTENCES)} categories')


✓ Benchmark loaded: 40 sentences across 8 categories


---

## Section 3: DeBERTa-v3 zero-shot classification

Using `MoritzLaurer/deberta-v3-large-zeroshot-v2.0`. This model was specifically fine-tuned for zero-shot classification on diverse NLI tasks and consistently outperforms BART-MNLI on benchmarks.

We compute both:
- **Multi-label scores** (`multi_label=True`): independent 0–1 score per category, used for the cross-method convergence and facade analyses
- **Forced-choice top label** (`multi_label=False`): single best-fitting category, used for the descriptive top-category distribution


In [7]:
# === LOAD DEBERTA ===
print('Loading DeBERTa-v3-large-zeroshot-v2.0 (≈1.5 GB download on first run)...')

deberta = pipeline(
    'zero-shot-classification',
    model='MoritzLaurer/deberta-v3-large-zeroshot-v2.0',
    device=0 if torch.cuda.is_available() else -1,
)
print('✓ DeBERTa loaded')


Loading DeBERTa-v3-large-zeroshot-v2.0 (≈1.5 GB download on first run)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

✓ DeBERTa loaded


In [8]:
# === DEBERTA: MULTI-LABEL ===
def deberta_classify(text, multi_label):
    """Returns dict of label->score, NaN if text invalid."""
    if not isinstance(text, str) or len(text.split()) < 3:
        return {l: np.nan for l in LABELS}
    try:
        # Truncate to 512 tokens worth (~ 400 words) defensively
        result = deberta(text[:1500], LABELS, multi_label=multi_label)
        return dict(zip(result['labels'], result['scores']))
    except Exception as e:
        print(f'  Error on text: {e}')
        return {l: np.nan for l in LABELS}

print('Running DeBERTa multi-label on 496 responses...')
ml_results = []
for txt in tqdm(df['text_combined'].tolist()):
    ml_results.append(deberta_classify(txt, multi_label=True))

ml_df = pd.DataFrame(ml_results)
ml_df.columns = [col(c, 'deberta_ml') for c in ml_df.columns]
df = pd.concat([df.reset_index(drop=True), ml_df.reset_index(drop=True)], axis=1)

print('\nDeBERTa multi-label means:')
for l in LABELS:
    side = 'prosocial' if l in PROSOCIAL_LABELS else 'dark'
    print(f'  {l:35s} ({side:9s}) {df[col(l, "deberta_ml")].mean():.3f}')


Running DeBERTa multi-label on 496 responses...


  0%|          | 0/496 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



DeBERTa multi-label means:
  deterrence and prevention           (prosocial) 0.556
  public safety and protection        (prosocial) 0.337
  rehabilitation and reform           (prosocial) 0.127
  proportional justice                (prosocial) 0.574
  societal condemnation               (prosocial) 0.569
  victim closure                      (prosocial) 0.221
  punishment and suffering            (dark     ) 0.670
  revenge and payback                 (dark     ) 0.217


In [9]:
# === DEBERTA: FORCED-CHOICE ===
print('Running DeBERTa forced-choice on 496 responses...')
fc_results = []
fc_top = []
for txt in tqdm(df['text_combined'].tolist()):
    scores = deberta_classify(txt, multi_label=False)
    fc_results.append(scores)
    if any(pd.isna(v) for v in scores.values()):
        fc_top.append(np.nan)
    else:
        fc_top.append(max(scores, key=scores.get))

fc_df = pd.DataFrame(fc_results)
fc_df.columns = [col(c, 'deberta_fc') for c in fc_df.columns]
df = pd.concat([df.reset_index(drop=True), fc_df.reset_index(drop=True)], axis=1)
df['deberta_fc_top'] = fc_top
df['deberta_ml_top'] = ml_df.idxmax(axis=1).str.replace('deberta_ml_', '').str.replace('_', ' ')

print('\nDeBERTa forced-choice top-category distribution:')
print(df['deberta_fc_top'].value_counts().to_string())
print('\nDeBERTa multi-label top-category distribution (argmax of multi-label scores):')
print(df['deberta_ml_top'].value_counts().to_string())


Running DeBERTa forced-choice on 496 responses...


  0%|          | 0/496 [00:00<?, ?it/s]


DeBERTa forced-choice top-category distribution:
deberta_fc_top
punishment and suffering        120
deterrence and prevention        98
societal condemnation            97
proportional justice             82
public safety and protection     35
rehabilitation and reform        34
revenge and payback              20
victim closure                   10

DeBERTa multi-label top-category distribution (argmax of multi-label scores):
deberta_ml_top
punishment and suffering        119
societal condemnation            98
deterrence and prevention        97
proportional justice             84
public safety and protection     35
rehabilitation and reform        33
revenge and payback              20
victim closure                   10


In [10]:
# === DEBERTA: AGGREGATES ===
df['deberta_ml_prosocial_mean'] = df[[col(l, 'deberta_ml') for l in PROSOCIAL_LABELS]].mean(axis=1)
df['deberta_ml_dark_mean']      = df[[col(l, 'deberta_ml') for l in DARK_LABELS]].mean(axis=1)
df['deberta_ml_prosocial_minus_dark'] = df['deberta_ml_prosocial_mean'] - df['deberta_ml_dark_mean']

df['deberta_fc_prosocial_mean'] = df[[col(l, 'deberta_fc') for l in PROSOCIAL_LABELS]].mean(axis=1)
df['deberta_fc_dark_mean']      = df[[col(l, 'deberta_fc') for l in DARK_LABELS]].mean(axis=1)
df['deberta_fc_prosocial_minus_dark'] = df['deberta_fc_prosocial_mean'] - df['deberta_fc_dark_mean']

print(f'DeBERTa multi-label aggregates:')
print(f'  prosocial mean: {df["deberta_ml_prosocial_mean"].mean():.3f}')
print(f'  dark mean:      {df["deberta_ml_dark_mean"].mean():.3f}')
print(f'  pro−dark gap:   {df["deberta_ml_prosocial_minus_dark"].mean():.3f}')
print()
print(f'DeBERTa forced-choice aggregates:')
print(f'  prosocial mean: {df["deberta_fc_prosocial_mean"].mean():.3f}')
print(f'  dark mean:      {df["deberta_fc_dark_mean"].mean():.3f}')
print(f'  pro−dark gap:   {df["deberta_fc_prosocial_minus_dark"].mean():.3f}')


DeBERTa multi-label aggregates:
  prosocial mean: 0.397
  dark mean:      0.444
  pro−dark gap:   -0.046

DeBERTa forced-choice aggregates:
  prosocial mean: 0.122
  dark mean:      0.134
  pro−dark gap:   -0.012


---

## Section 4: Claude Haiku 4.5 zero-shot classification

Independent classifier using Claude Haiku 4.5 with structured JSON output. We send each response twice — once asking for forced-choice (single category + confidence), once asking for multi-label scores (independent 0–1 per category).

**Parallelization:** 8 concurrent workers. With ~496 responses × 2 modes + 40 benchmark × 2 = ~1070 calls, total time is ~3–5 min and cost is ~$0.50–$0.80.

**Robustness:** JSON parsing has fallback logic. On malformed output, we retry once; if still malformed, that response gets NaN scores for that mode. Rate limits are handled by the SDK with exponential backoff.


In [11]:
# === CLAUDE PROMPTS ===
# Restructured: descriptions appear as separate guidance, not as part of category names.
# Includes explicit instruction to return ONLY the short category name in the JSON output.

CANONICAL_CATEGORIES = '\n'.join(f'  - {l}' for l in LABELS)

CATEGORY_GUIDANCE = '''Category meanings (for your interpretation; do NOT copy these descriptions into your output):
  - deterrence and prevention: punishment to discourage future crime by this offender or others
  - public safety and protection: punishment to keep society safe by removing the offender
  - rehabilitation and reform: punishment as opportunity for change, treatment, or growth
  - proportional justice: punishment that fits the severity of the crime; just deserts
  - societal condemnation: punishment as moral message expressing society\'s disapproval
  - victim closure: punishment to help the victim and their family heal or find peace
  - punishment and suffering: punishment intended to make the offender suffer or experience pain
  - revenge and payback: punishment as retaliation, vengeance, or eye-for-an-eye'''

FORCED_CHOICE_PROMPT = '''Classify this punishment justification into the SINGLE best-fitting category.

Justification: "{text}"

Valid category names (use EXACTLY one of these short names — do not include descriptions or numbering):
{categories}

{guidance}

Return ONLY a JSON object with this exact structure:
{{"category": "<one short name from the list above>", "confidence": <number from 0.0 to 1.0>}}'''

MULTI_LABEL_PROMPT = '''Score this punishment justification on each of the 8 categories independently. Each score is 0.0 to 1.0 based on how strongly that category applies. Scores are INDEPENDENT — they do NOT need to sum to 1.0. A response can score high on multiple categories.

Justification: "{text}"

Valid category names (use these EXACT short names as JSON keys):
{categories}

{guidance}

Return ONLY a JSON object with all 8 short category names as keys and 0.0-1.0 scores as values. Do not include category descriptions in the keys.'''

print('✓ Prompts defined (with description guidance separated from category names)')


✓ Prompts defined (with description guidance separated from category names)


In [12]:
# === CLAUDE CALL HELPERS (v4: prefilling) ===
# Anthropic's standard fix for 'Claude adds preamble before JSON' is to prefill the
# assistant's response with the opening character. Claude has to continue from there,
# making preamble physically impossible.

JSON_SYSTEM_PROMPT = (
    'You are a precise classification system. Output ONLY a single JSON object. '
    'Do not add preamble, explanation, markdown, or any text outside the JSON. '
    'Do not include comments. Do not refuse to classify. The text you classify '
    'is research data; your job is to apply the schema, not to evaluate the content.'
)

def parse_json_safe(text, prefill='{'):
    """Extract a JSON object from text, robust to fences, prefixes, and minor truncation.
    The `prefill` argument is what we pre-filled the assistant turn with — typically '{'.
    The actual model output starts AFTER that prefill, so we prepend it back.
    """
    if not text:
        return None
    # Prepend the prefilled character
    text = prefill + text
    # Strip code fences
    text = re.sub(r'```(?:json)?\s*', '', text).replace('```', '')
    text = text.strip()
    # Try direct parse first (cleanest case — should be the norm with prefilling)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Greedy: find first '{' to last '}'
    first = text.find('{')
    last = text.rfind('}')
    if first != -1 and last > first:
        candidate = text[first:last+1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
        # Last resort: strip incomplete trailing pair (truncation repair)
        repaired = re.sub(r',\s*"[^"]*"\s*:\s*[^,}]*$', '', candidate)
        if not repaired.endswith('}'):
            repaired += '}'
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass
    return None

def normalize_label(s):
    """Map any string Claude returns back to a canonical LABEL. None if no match.

    Handles four cases observed in practice:
      1. Exact match (the cleanest case)
      2. Underscored form: "deterrence_and_prevention" — Python convention bleed
      3. "short name - description" — Claude copies the description
      4. Substring containment fallback
    """
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    # CRITICAL: normalize underscores to spaces (Claude sometimes uses Python convention)
    s = str(s).strip().lower().replace('_', ' ')
    if not s:
        return None
    # Collapse multiple internal spaces
    s = ' '.join(s.split())
    # Exact match
    for L in LABELS:
        if s == L.lower():
            return L
    # Prefix-before-' - ' match
    if ' - ' in s:
        prefix = s.split(' - ')[0].strip()
        for L in LABELS:
            if prefix == L.lower():
                return L
    # Substring fallback
    for L in LABELS:
        if L.lower() in s:
            return L
    return None

# Diagnostic capture: log first 5 failures to a global list for inspection
FAILURE_LOG = []

def claude_call(prompt, max_tokens=800, max_retries=4):
    """Single Claude call with prefilling, retries on parse failure or rate limit.

    Prefills the assistant turn with '{' so the response is forced to be JSON.
    """
    last_raw = None
    for attempt in range(max_retries + 1):
        try:
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=max_tokens,
                system=JSON_SYSTEM_PROMPT,
                messages=[
                    {'role': 'user', 'content': prompt},
                    {'role': 'assistant', 'content': '{'},  # ← prefill
                ],
            )
            content = response.content[0].text
            last_raw = content
            parsed = parse_json_safe(content, prefill='{')
            if parsed is not None:
                return parsed
            time.sleep(0.5)
        except anthropic.RateLimitError:
            time.sleep(2 ** attempt)
        except Exception:
            if attempt == max_retries:
                break
            time.sleep(1)
    # Log first 5 failures for diagnostic
    if len(FAILURE_LOG) < 5 and last_raw is not None:
        FAILURE_LOG.append({'prompt_first120': prompt[:120], 'raw_response': last_raw[:500]})
    return None

def claude_forced_choice(text):
    if not isinstance(text, str) or len(text.split()) < 3:
        return {'category': None, 'confidence': np.nan}
    prompt = FORCED_CHOICE_PROMPT.format(
        text=text[:1500], categories=CANONICAL_CATEGORIES, guidance=CATEGORY_GUIDANCE)
    result = claude_call(prompt, max_tokens=200)  # FC JSON is short
    if result is None or 'category' not in result:
        return {'category': None, 'confidence': np.nan}
    result['category'] = normalize_label(result.get('category'))
    try:
        result['confidence'] = float(result.get('confidence', np.nan))
    except (ValueError, TypeError):
        result['confidence'] = np.nan
    return result

def claude_multi_label(text):
    if not isinstance(text, str) or len(text.split()) < 3:
        return {l: np.nan for l in LABELS}
    prompt = MULTI_LABEL_PROMPT.format(
        text=text[:1500], categories=CANONICAL_CATEGORIES, guidance=CATEGORY_GUIDANCE)
    result = claude_call(prompt, max_tokens=800)  # Multi-label needs headroom
    if result is None:
        return {l: np.nan for l in LABELS}
    out = {l: np.nan for l in LABELS}
    for k, v in result.items():
        canonical = normalize_label(k)
        if canonical is None:
            continue
        try:
            out[canonical] = float(v)
        except (ValueError, TypeError):
            out[canonical] = np.nan
    return out

print('✓ Helpers defined (v5: prefilling + underscore-aware normalize_label + FAILURE_LOG)')


✓ Helpers defined (v5: prefilling + underscore-aware normalize_label + FAILURE_LOG)


In [13]:
# === CLAUDE FORCED-CHOICE (PARALLEL) ===
print(f'Running Claude forced-choice on {len(df)} responses (8 workers)...')
fc_results = [None] * len(df)
texts = df['text_combined'].tolist()

with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(claude_forced_choice, t): i for i, t in enumerate(texts)}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        i = futures[fut]
        fc_results[i] = fut.result()

df['claude_fc_top'] = [r.get('category') if r else None for r in fc_results]
df['claude_fc_confidence'] = [r.get('confidence', np.nan) if r else np.nan for r in fc_results]

n_valid_fc = df['claude_fc_top'].notna().sum()
print(f'\n✓ Forced-choice complete ({n_valid_fc}/{len(df)} valid)')
print('\nClaude forced-choice top-category distribution:')
print(df['claude_fc_top'].value_counts().to_string())


Running Claude forced-choice on 496 responses (8 workers)...


  0%|          | 0/496 [00:00<?, ?it/s]


✓ Forced-choice complete (496/496 valid)

Claude forced-choice top-category distribution:
claude_fc_top
public safety and protection    151
rehabilitation and reform       122
proportional justice             95
deterrence and prevention        91
victim closure                   19
punishment and suffering         11
revenge and payback               5
societal condemnation             2


In [14]:
# === CLAUDE MULTI-LABEL (PARALLEL) ===
print(f'Running Claude multi-label on {len(df)} responses (8 workers)...')
ml_results = [None] * len(df)

with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(claude_multi_label, t): i for i, t in enumerate(texts)}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        i = futures[fut]
        ml_results[i] = fut.result()

ml_df = pd.DataFrame(ml_results)
ml_df.columns = [col(c, 'claude_ml') for c in ml_df.columns]
df = pd.concat([df.reset_index(drop=True), ml_df.reset_index(drop=True)], axis=1)

# Top category from multi-label
df['claude_ml_top'] = ml_df.idxmax(axis=1).str.replace('claude_ml_', '').str.replace('_', ' ')

print(f'\n✓ Multi-label complete')
print('\nClaude multi-label means:')
for l in LABELS:
    side = 'prosocial' if l in PROSOCIAL_LABELS else 'dark'
    print(f'  {l:35s} ({side:9s}) {df[col(l, "claude_ml")].mean():.3f}')


Running Claude multi-label on 496 responses (8 workers)...


  0%|          | 0/496 [00:00<?, ?it/s]


✓ Multi-label complete

Claude multi-label means:
  deterrence and prevention           (prosocial) 0.538
  public safety and protection        (prosocial) 0.572
  rehabilitation and reform           (prosocial) 0.295
  proportional justice                (prosocial) 0.535
  societal condemnation               (prosocial) 0.440
  victim closure                      (prosocial) 0.238
  punishment and suffering            (dark     ) 0.240
  revenge and payback                 (dark     ) 0.194


In [15]:
# === CLAUDE: AGGREGATES ===
df['claude_ml_prosocial_mean'] = df[[col(l, 'claude_ml') for l in PROSOCIAL_LABELS]].mean(axis=1)
df['claude_ml_dark_mean']      = df[[col(l, 'claude_ml') for l in DARK_LABELS]].mean(axis=1)
df['claude_ml_prosocial_minus_dark'] = df['claude_ml_prosocial_mean'] - df['claude_ml_dark_mean']

# Forced-choice 'aggregate' = proportion of cases assigned to each side
df['claude_fc_is_prosocial'] = df['claude_fc_top'].isin(PROSOCIAL_LABELS).astype(int)
df['claude_fc_is_dark']      = df['claude_fc_top'].isin(DARK_LABELS).astype(int)

print(f'Claude multi-label aggregates:')
print(f'  prosocial mean: {df["claude_ml_prosocial_mean"].mean():.3f}')
print(f'  dark mean:      {df["claude_ml_dark_mean"].mean():.3f}')
print(f'  pro−dark gap:   {df["claude_ml_prosocial_minus_dark"].mean():.3f}')
print()
print(f'Claude forced-choice (top-category):')
print(f'  % assigned prosocial: {df["claude_fc_is_prosocial"].mean()*100:.1f}%')
print(f'  % assigned dark:      {df["claude_fc_is_dark"].mean()*100:.1f}%')


Claude multi-label aggregates:
  prosocial mean: 0.436
  dark mean:      0.217
  pro−dark gap:   0.219

Claude forced-choice (top-category):
  % assigned prosocial: 96.8%
  % assigned dark:      3.2%


---

## Section 5: A1 synthetic benchmark — head-to-head comparison

Run both classifiers on the 40 hand-crafted unambiguous sentences. Reports top-1, top-2, top-3 accuracy, mean true-label rank, and confusion patterns.

**BART baseline (v2 pipeline, for comparison):** 77.5% top-1, 92.5% top-2, 95.0% top-3.


In [16]:
# === RUN BENCHMARK ON DEBERTA ===
print('Running A1 benchmark on DeBERTa...')
deberta_bench = []
for true_label, sentences in BENCHMARK_SENTENCES.items():
    for sent in sentences:
        result = deberta(sent, LABELS, multi_label=True)
        pred_label = result['labels'][0]
        pred_score = result['scores'][0]
        true_idx = result['labels'].index(true_label)
        true_score = result['scores'][true_idx]
        true_rank = true_idx + 1
        deberta_bench.append({
            'classifier': 'DeBERTa-v3',
            'true_label': true_label,
            'predicted_label': pred_label,
            'correct_top1': pred_label == true_label,
            'true_label_score': true_score,
            'true_label_rank': true_rank,
            'pred_confidence': pred_score,
            'sentence': sent,
        })
deberta_bench_df = pd.DataFrame(deberta_bench)
print(f'✓ Done ({len(deberta_bench_df)} sentences)')


Running A1 benchmark on DeBERTa...
✓ Done (40 sentences)


In [17]:
# === RUN BENCHMARK ON CLAUDE ===
print('Running A1 benchmark on Claude (parallel)...')

# Flatten benchmark
flat = [(label, sent) for label, sents in BENCHMARK_SENTENCES.items() for sent in sents]
claude_bench_results = [None] * len(flat)

with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {}
    for i, (_, sent) in enumerate(flat):
        futures[ex.submit(claude_multi_label, sent)] = i
    for fut in tqdm(as_completed(futures), total=len(futures)):
        i = futures[fut]
        claude_bench_results[i] = fut.result()

claude_bench = []
n_failed = 0
for (true_label, sent), scores in zip(flat, claude_bench_results):
    if scores is None or all(pd.isna(v) for v in scores.values()):
        # Record the failure rather than dropping the row — keeps the benchmark honest
        claude_bench.append({
            'classifier': 'Claude Haiku 4.5',
            'true_label': true_label,
            'predicted_label': None,
            'correct_top1': False,
            'true_label_score': np.nan,
            'true_label_rank': np.nan,
            'pred_confidence': np.nan,
            'sentence': sent,
        })
        n_failed += 1
        continue
    sorted_labels = sorted(scores, key=lambda k: scores[k] if not pd.isna(scores[k]) else -1, reverse=True)
    pred_label = sorted_labels[0]
    pred_score = scores[pred_label]
    true_score = scores.get(true_label, np.nan)
    true_rank = sorted_labels.index(true_label) + 1
    claude_bench.append({
        'classifier': 'Claude Haiku 4.5',
        'true_label': true_label,
        'predicted_label': pred_label,
        'correct_top1': pred_label == true_label,
        'true_label_score': true_score,
        'true_label_rank': true_rank,
        'pred_confidence': pred_score,
        'sentence': sent,
    })

claude_bench_df = pd.DataFrame(claude_bench)
if n_failed > 0:
    print(f'⚠ {n_failed} of {len(flat)} sentences had unparseable Claude responses (recorded as failures)')
else:
    print(f'✓ All {len(flat)} sentences classified successfully')
print(f'  Total benchmark rows for Claude: {len(claude_bench_df)}')


Running A1 benchmark on Claude (parallel)...


  0%|          | 0/40 [00:00<?, ?it/s]

✓ All 40 sentences classified successfully
  Total benchmark rows for Claude: 40


In [18]:
# === HEAD-TO-HEAD SUMMARY ===
def summarize(bench_df, name):
    n = len(bench_df)
    if n == 0:
        return None
    return {
        'classifier': name,
        'n': n,
        'top1': bench_df['correct_top1'].mean(),
        'top2': (bench_df['true_label_rank'] <= 2).mean(),
        'top3': (bench_df['true_label_rank'] <= 3).mean(),
        'mean_rank': bench_df['true_label_rank'].mean(),
        'mean_true_score': bench_df['true_label_score'].mean(),
    }

summary_rows = [
    {'classifier': 'BART-large-MNLI (v2 historical)', 'n': 40, 'top1': 0.775, 'top2': 0.925,
     'top3': 0.950, 'mean_rank': 1.40, 'mean_true_score': 0.932},
    summarize(deberta_bench_df, 'DeBERTa-v3-large'),
    summarize(claude_bench_df, 'Claude Haiku 4.5'),
]
summary_df = pd.DataFrame([r for r in summary_rows if r is not None])
print('\n=== A1 BENCHMARK HEAD-TO-HEAD ===\n')
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print(f'BART → DeBERTa Δtop1:  {summary_df.iloc[1]["top1"] - 0.775:+.3f}')
print(f'BART → Claude Δtop1:   {summary_df.iloc[2]["top1"] - 0.775:+.3f}')



=== A1 BENCHMARK HEAD-TO-HEAD ===

                     classifier  n  top1  top2  top3  mean_rank  mean_true_score
BART-large-MNLI (v2 historical) 40 0.775 0.925 0.950      1.400            0.932
               DeBERTa-v3-large 40 0.875 0.950 0.975      1.200            0.889
               Claude Haiku 4.5 40 1.000 1.000 1.000      1.000            0.921

BART → DeBERTa Δtop1:  +0.100
BART → Claude Δtop1:   +0.225


In [19]:
# === PER-CATEGORY ACCURACY ===
print('=== PER-CATEGORY TOP-1 ACCURACY ===\n')
for label in LABELS:
    d_acc = deberta_bench_df[deberta_bench_df['true_label'] == label]['correct_top1'].mean()
    c_acc = claude_bench_df[claude_bench_df['true_label'] == label]['correct_top1'].mean() if len(claude_bench_df) else np.nan
    print(f'  {label:35s} DeBERTa={d_acc:.0%}  Claude={c_acc:.0%}')

# Combined benchmark df for export
all_bench_df = pd.concat([deberta_bench_df, claude_bench_df], ignore_index=True)
all_bench_df.to_csv('a1_benchmark_validation.csv', index=False)
print('\n✓ Saved: a1_benchmark_validation.csv')


=== PER-CATEGORY TOP-1 ACCURACY ===

  deterrence and prevention           DeBERTa=100%  Claude=100%
  public safety and protection        DeBERTa=100%  Claude=100%
  rehabilitation and reform           DeBERTa=80%  Claude=100%
  proportional justice                DeBERTa=100%  Claude=100%
  societal condemnation               DeBERTa=100%  Claude=100%
  victim closure                      DeBERTa=60%  Claude=100%
  punishment and suffering            DeBERTa=100%  Claude=100%
  revenge and payback                 DeBERTa=60%  Claude=100%

✓ Saved: a1_benchmark_validation.csv


In [20]:
# === CONFUSION PATTERNS (handles failure rows) ===
print('=== DEBERTA MISCLASSIFICATIONS ===\n')
miss_d = deberta_bench_df[~deberta_bench_df['correct_top1']]
if len(miss_d) == 0:
    print('  None — perfect accuracy!\n')
else:
    for _, r in miss_d.iterrows():
        true_l = (r['true_label'] or '')[:25]
        pred_l = (r['predicted_label'] or 'FAILED')[:25]
        print(f'  TRUE: {true_l:25s} → PRED: {pred_l:25s}')
        print(f'    "{r["sentence"]}"')

print('\n=== CLAUDE MISCLASSIFICATIONS / FAILURES ===\n')
miss_c = claude_bench_df[~claude_bench_df['correct_top1']]
n_failures = miss_c['predicted_label'].isna().sum()
n_real_miss = len(miss_c) - n_failures
print(f'  {n_real_miss} real misclassifications, {n_failures} JSON parse failures\n')

if len(miss_c) == 0:
    print('  None — perfect accuracy!\n')
else:
    for _, r in miss_c.iterrows():
        true_l = (r['true_label'] or '')[:25]
        pred = r['predicted_label']
        if pd.isna(pred) or pred is None:
            pred_l = 'FAILED (no JSON)'
        else:
            pred_l = str(pred)[:25]
        print(f'  TRUE: {true_l:25s} → PRED: {pred_l:25s}')
        print(f'    "{r["sentence"]}"')

# Diagnostic: show captured raw failures if any
if len(FAILURE_LOG) > 0:
    print(f'\n=== DIAGNOSTIC: First {len(FAILURE_LOG)} raw failure responses ===')
    for j, fail in enumerate(FAILURE_LOG):
        print(f'\n--- Failure {j+1} ---')
        print(f'  Prompt start: {fail["prompt_first120"]}')
        print(f'  Raw response: {fail["raw_response"]}')


=== DEBERTA MISCLASSIFICATIONS ===

  TRUE: rehabilitation and reform → PRED: punishment and suffering 
    "Prison should give this person a chance to turn their life around."
  TRUE: victim closure            → PRED: proportional justice     
    "The victim's family deserves to see justice served."
  TRUE: victim closure            → PRED: punishment and suffering 
    "Seeing the offender punished can help the victims begin to heal."
  TRUE: revenge and payback       → PRED: proportional justice     
    "He should get a taste of his own medicine."
  TRUE: revenge and payback       → PRED: punishment and suffering 
    "This is payback for the suffering he caused."

=== CLAUDE MISCLASSIFICATIONS / FAILURES ===

  0 real misclassifications, 0 JSON parse failures

  None — perfect accuracy!



---

## Section 6: Export `02_classification_features.csv`

**New columns added in this notebook (≈ 40):**
- 8 DeBERTa multi-label scores (`deberta_ml_*`)
- 8 DeBERTa forced-choice scores (`deberta_fc_*`)
- 2 DeBERTa top labels (`deberta_ml_top`, `deberta_fc_top`)
- 6 DeBERTa aggregates (prosocial mean, dark mean, gap × 2 modes)
- 8 Claude multi-label scores (`claude_ml_*`)
- 1 Claude forced-choice top + 1 confidence (`claude_fc_top`, `claude_fc_confidence`)
- 1 Claude multi-label top (`claude_ml_top`)
- 5 Claude aggregates (prosocial mean, dark mean, gap, fc-is-prosocial, fc-is-dark)


In [21]:
# === EXPORT ===
out_path = '02_classification_features.csv'
df.to_csv(out_path, index=False)
summary_df.to_csv('a1_benchmark_summary.csv', index=False)

print(f'✓ Exported: {out_path}')
print(f'  Rows: {len(df)}, Total columns: {len(df.columns)}')
print()
print('New classification columns in this notebook:')
new_cols = [c for c in df.columns if c.startswith(('deberta_', 'claude_'))]
for c in new_cols:
    print(f'  {c}')
print(f'\nTotal new: {len(new_cols)}')


✓ Exported: 02_classification_features.csv
  Rows: 496, Total columns: 242

New classification columns in this notebook:
  deberta_ml_proportional_justice
  deberta_ml_victim_closure
  deberta_ml_public_safety_and_protection
  deberta_ml_punishment_and_suffering
  deberta_ml_rehabilitation_and_reform
  deberta_ml_societal_condemnation
  deberta_ml_deterrence_and_prevention
  deberta_ml_revenge_and_payback
  deberta_fc_proportional_justice
  deberta_fc_victim_closure
  deberta_fc_public_safety_and_protection
  deberta_fc_rehabilitation_and_reform
  deberta_fc_punishment_and_suffering
  deberta_fc_societal_condemnation
  deberta_fc_deterrence_and_prevention
  deberta_fc_revenge_and_payback
  deberta_fc_top
  deberta_ml_top
  deberta_ml_prosocial_mean
  deberta_ml_dark_mean
  deberta_ml_prosocial_minus_dark
  deberta_fc_prosocial_mean
  deberta_fc_dark_mean
  deberta_fc_prosocial_minus_dark
  claude_fc_top
  claude_fc_confidence
  claude_ml_deterrence_and_prevention
  claude_ml_public_saf

In [22]:
# === DOWNLOAD ===
files.download('02_classification_features.csv')
files.download('a1_benchmark_validation.csv')
files.download('a1_benchmark_summary.csv')
print('✓ Three downloads triggered. Save these — main features feeds Notebook 3.')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Three downloads triggered. Save these — main features feeds Notebook 3.


---

## Next step: Notebook 3

Move to **`03_embeddings_and_similarity.ipynb`** which will:
1. Load `02_classification_features.csv`
2. Generate embeddings with `BAAI/bge-large-en-v1.5` (replaces all-mpnet-base-v2)
3. Generate embeddings with `voyage-4-large` via the Voyage API
4. Compute prototype similarity for both embedding sets across **four prototype variants** (original, formal, colloquial, retribution-removed)
5. Run prototype sensitivity analysis (B1/B2/B3)
6. Export `03_embedding_features.csv`

**Notebook 3 requires GPU** for BGE; Voyage is API-based so it runs over network.
